# JASSIM AI — Kaggle GPU evaluation (free-session mode)

دفتر اختبار مؤقت للنموذج المفتوح على GPU المجاني المتاح في Kaggle. يستخدم Commit ثابتاً وإصدار Ollama ثابتاً للتحقق القابل لإعادة الإنتاج. لا يحتاج إلى مفاتيح API أو أسرار أو حسابات خارجية مرتبطة بالمشروع.

**حدود الخصوصية:** يرسل هذا الدفتر بيانات اختبار اصطناعية فقط إلى Ollama على `127.0.0.1`. لا ينشئ نفقاً عاماً، ولا يحدّث `dashboard/connection.json`، ولا يرفع المحادثات أو قاعدة البيانات إلى GitHub/Drive. لا تستخدمه مع رسائل حقيقية.

**الاستمرارية:** جلسات Kaggle وموارد GPU المجانية مؤقتة وخاضعة للحصة والتوفر؛ هذا ليس خادماً دائماً ولا يثبت تكامل Android من طرف إلى طرف.


## 1. تحقق من بيئة GPU والاتصال


In [ ]:
import os, subprocess, sys, json, time

gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'No NVIDIA GPU detected in this session.')
if gpu.returncode != 0:
    raise RuntimeError('Enable a Kaggle GPU accelerator in Notebook Settings, then restart the session.')
print('GPU session detected. No secrets are required.')


## 2. احصل على كود الفرع المحدد


In [ ]:
from pathlib import Path
PROJECT = Path('/kaggle/working/JASSIM-AI-Android')
PINNED_COMMIT = '3a88523fb649f92f1f812476268725255ff85c1e'
REPOSITORY = 'https://github.com/stevennpllk-crypto/-JASSIM-AI-Android.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY, str(PROJECT)], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'fetch', '--depth', '1', 'origin', PINNED_COMMIT], check=True)
subprocess.run(['git', '-C', str(PROJECT), 'checkout', '--detach', PINNED_COMMIT], check=True)
BACKEND = PROJECT / 'backend'
assert (BACKEND / 'local_model_benchmark.py').is_file()
sys.path.insert(0, str(BACKEND))
actual_commit = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == PINNED_COMMIT, f'Unexpected source revision: {actual_commit}'
print('Pinned source commit:', actual_commit)


## 3. ثبّت Ollama وشغّله محلياً


In [ ]:
from pathlib import Path
import hashlib
import urllib.request

# Pin the official installer asset and verify its published SHA-256 before execution.
OLLAMA_VERSION = '0.40.16'
INSTALLER_URL = 'https://github.com/ollama/ollama/releases/download/v0.40.16/install.sh'
INSTALLER_SHA256 = '25f64b810b947145095956533e1bdf56eacea2673c55a7e586be4515fc882c9f'
INSTALLER = Path('/kaggle/working/ollama-install.sh')
download = subprocess.run(
    ['curl', '-fL', '--proto', '=https', '--tlsv1.2', INSTALLER_URL, '-o', str(INSTALLER)],
    capture_output=True, text=True
)
if download.returncode != 0:
    raise RuntimeError('Pinned Ollama installer download failed: ' + (download.stderr or '')[-1500:])
actual_sha256 = hashlib.sha256(INSTALLER.read_bytes()).hexdigest()
if actual_sha256 != INSTALLER_SHA256:
    raise RuntimeError(f'Installer checksum mismatch: {actual_sha256}')
install_env = os.environ.copy()
install_env['OLLAMA_VERSION'] = OLLAMA_VERSION
install = subprocess.run(['bash', str(INSTALLER)], env=install_env, capture_output=True, text=True)
print((install.stdout or '')[-2000:])
if install.returncode != 0:
    print((install.stderr or '')[-3000:])
    raise RuntimeError('Pinned Ollama installation failed')

# Bind the model server to loopback only. The local client below bypasses proxies
# and refuses redirects, so benchmark prompts cannot be forwarded by proxy settings.
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
os.environ['NO_PROXY'] = '127.0.0.1,localhost,::1'
os.environ['no_proxy'] = '127.0.0.1,localhost,::1'

class _NoRedirectHandler(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None

LOCAL_OPENER = urllib.request.build_opener(
    urllib.request.ProxyHandler({}),
    _NoRedirectHandler()
)

ollama_log = open('/kaggle/working/ollama.log', 'w')
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
for attempt in range(60):
    try:
        with LOCAL_OPENER.open('http://127.0.0.1:11434/api/tags', timeout=3) as response:
            assert response.status == 200
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Ollama did not become ready; inspect /kaggle/working/ollama.log')
print('Ollama is listening on loopback only; proxy and redirects are disabled.')


## 4. نزّل نموذج الاختبار المفتوح

يتم تنزيل أوزان النموذج من سجل Ollama إلى جلسة Kaggle المؤقتة. لا يتم تنزيل محادثات المستخدمين. اخترنا 4B كبداية لتقليل استهلاك الذاكرة.


In [ ]:
MODEL = 'qwen3.5:4b'
pull = subprocess.run(['ollama', 'pull', MODEL], text=True)
if pull.returncode != 0:
    raise RuntimeError('Model download failed; check Kaggle internet availability and GPU quota.')
print('Model weights downloaded into this temporary runtime:', MODEL)

# Force one synthetic local inference before benchmarking, then verify that
# Ollama actually assigned the model to an NVIDIA GPU (not CPU-only fallback).
warmup_payload = json.dumps({
    'model': MODEL,
    'messages': [{'role': 'user', 'content': 'أجب بكلمة واحدة: مرحباً'}],
    'stream': False,
    'keep_alive': '5m',
    'options': {'temperature': 0, 'num_predict': 8, 'num_ctx': 4096},
}).encode('utf-8')
warmup_request = urllib.request.Request(
    'http://127.0.0.1:11434/api/chat',
    data=warmup_payload,
    headers={'Content-Type': 'application/json'},
    method='POST',
)
with LOCAL_OPENER.open(warmup_request, timeout=240) as response:
    warmup_result = json.loads(response.read().decode('utf-8'))
assert (warmup_result.get('message') or {}).get('content', '').strip(), 'Warmup inference was empty'

processor = subprocess.run(['ollama', 'ps'], capture_output=True, text=True, check=True)
print(processor.stdout)
if 'GPU' not in processor.stdout.upper():
    raise RuntimeError('Ollama did not report GPU placement. Stop here; do not label this a GPU benchmark.')
print('PASS: Ollama reports GPU placement for the loaded model.')


## 5. شغّل اختبارات JASSIM الاصطناعية

الاختبارات تقيس العربية والتونسية والحساب والمنطق واتباع التعليمات واسترجاع سياق تجريبي. النتيجة ليست تقييماً شاملاً ولا مقارنة نهائية مع Gemini/ChatGPT.


In [ ]:
from local_model_benchmark import run_benchmark
import adaptive
BASE_URL = 'http://127.0.0.1:11434'
report = run_benchmark(BASE_URL, MODEL, adaptive.EVALUATION_CASES, timeout=240)
summary = {k:v for k,v in report.items() if k != 'cases'}
print('Completed inference calls:', report['completed'], '/', report['case_count'])
print('Correct answers:', report['passed'], '/', report['completed'])
if report['completed'] != report['case_count']:
    failed_calls = [case for case in report['cases'] if 'error' in case]
    raise RuntimeError(f"Incomplete inference run: {len(failed_calls)} failed calls. Inspect report before drawing quality conclusions.")
print(json.dumps(summary,ensure_ascii=False,indent=2))
output = Path('/kaggle/working/jassim-kaggle-benchmark.json')
output.write_text(json.dumps({'policy':'synthetic_prompts_only_loopback_ollama','report':report},ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved benchmark artifact:', output)


## 6. تحقق مستقل من أن Ollama يجيب فعلياً


In [ ]:
payload = json.dumps({'model':MODEL,'messages':[{'role':'user','content':'أجب بكلمة واحدة فقط: ما عاصمة تونس؟'}],
                      'stream':False,'options':{'temperature':0,'num_predict':32,'num_ctx':4096}}).encode('utf-8')
req = urllib.request.Request(BASE_URL+'/api/chat',data=payload,headers={'Content-Type':'application/json'},method='POST')
with LOCAL_OPENER.open(req,timeout=240) as response:
    live = json.loads(response.read().decode('utf-8'))
answer = live.get('message',{}).get('content','')
print('Model:',live.get('model'))
print('Synthetic prompt answer:',answer)
assert answer.strip(), 'Ollama returned an empty answer'
print('PASS: a real inference response was produced by the local Ollama process.')


## 7. ما الذي أثبتناه وما الذي لم نثبته؟

- عند نجاح الخلايا: ثبت تشغيل نموذج فعلي داخل جلسة GPU الحالية وقياسه على الاختبارات الاصطناعية.
- لم نفتح أي endpoint عام ولم نربط تطبيق Android بهذه الجلسة.
- لم نختبر تسجيل الدخول أو تاريخ المحادثات من APK.
- لا تحفظ أسراراً أو محادثات حقيقية في هذا الدفتر. نزّل ملف JSON من مخرجات الجلسة إن أردت الاحتفاظ بالنتائج.
- أوقف الجلسة بعد انتهاء القياس لتجنّب استهلاك الحصة المجانية.
